## Wind Power in the US

In [1]:
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages 

In [2]:
# getting the US census data

url_0 = "https://www2.census.gov/geo/tiger/GENZ2024/shp/" 
url_county = "cb_2024_us_county_500k.zip"
url_state = "cb_2024_us_state_500k.zip" 


## code for county level data 

counties = gpd.read_file(url_0+url_county)
nc_counties = counties.loc[counties["STATEFP"] == "37"].to_crs(6543).copy()
nc_counties = nc_counties.sort_values("NAME").reset_index(drop=True)

In [16]:
# Read data

wind = gpd.read_file("data/wind_turbines.geojson")
wind = wind.to_crs(epsg=6543)
wind = wind[["t_state","p_name","geometry"]].copy()

# spatial join

wind_by_county = gpd.sjoin(wind, nc_counties, predicate='within')
wind_by_county

,t_state,p_name,geometry,index_right,STATEFP,COUNTYFP,COUNTYNS,GEOIDFQ,GEOID,NAME,NAMELSAD,STUSPS,STATE_NAME,LSAD,ALAND,AWATER
1220,NC,Amazon Wind Farm US East,POINT (2749500.194 948859.331),71,37,143,01025841,0500000US37143,37143,Perquimans,Perquimans County,NC,North Carolina,06,640173980,211755191
1221,NC,Amazon Wind Farm US East,POINT (2766474.571 953488.142),69,37,139,01026330,0500000US37139,37139,Pasquotank,Pasquotank County,NC,North Carolina,06,587610261,161751355
1222,NC,Amazon Wind Farm US East,POINT (2768534.22 948946.345),69,37,139,01026330,0500000US37139,37139,Pasquotank,Pasquotank County,NC,North Carolina,06,587610261,161751355
1223,NC,Amazon Wind Farm US East,POINT (2747874.717 933115.951),71,37,143,01025841,0500000US37143,37143,Perquimans,Perquimans County,NC,North Carolina,06,640173980,211755191
1224,NC,Amazon Wind Farm US East,POINT (2744305.468 934236.194),71,37,143,01025841,0500000US37143,37143,Perquimans,Perquimans County,NC,North Carolina,06,640173980,211755191
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
66367,NC,Timbermill Wind,POINT (2706664.54 900173.525),20,37,041,01025822,0500000US37041,37041,Chowan,Chowan County,NC,North Carolina,06,447179939,157966156
66368,NC,Timbermill Wind,POINT (2705444.578 867686.892),20,37,041,01025822,0500000US37041,37041,Chowan,Chowan County,NC,North Carolina,06,447179939,157966156
66369,NC,Timbermill Wind,POINT (2714104.835 876793.83),20,37,041,01025822,0500000US37041,37041,Chowan,Chowan County,NC,North Carolina,06,447179939,157966156
77711,NC,unknown Watauga County,POINT (1205770.058 907038.161),94,37,189,01008594,0500000US37189,37189,Watauga,Watauga County,NC,North Carolina,06,809214845,2271720


In [17]:
# wind count in each county

wind_count = wind_by_county["NAME"].value_counts()

In [18]:
# creating new table

wind_count_table = wind_count.reset_index()
wind_count_table.columns = ["NAME", "Wind Turbines"]
wind_count_table

,NAME,Wind Turbines
0,Perquimans,54
1,Pasquotank,50
2,Chowan,45
3,Watauga,3


In [19]:
# add new column

wind_by_county = nc_counties.merge(wind_count_table, on="NAME", how="left")
wind_by_county["Wind Turbines"] = wind_by_county["Wind Turbines"].fillna(0)
wind_by_county["Wind Turbines"] = wind_by_county["Wind Turbines"].astype(int)

# filtering for only needed columns

wind_by_county = wind_by_county[["NAME", "geometry", "Wind Turbines"]].copy()
wind_by_county

,NAME,geometry,Wind Turbines
0,Alamance,"POLYGON ((1839468.72 776874.593, 1839558.964 7...",0
1,Alexander,"POLYGON ((1305343.411 764119.949, 1307325.574 ...",0
2,Alleghany,"POLYGON ((1308931.907 1036329.442, 1317647.375...",0
3,Anson,"POLYGON ((1607201.434 423476.314, 1607481.922 ...",0
4,Ashe,"POLYGON ((1193571.583 980437.246, 1194562.461 ...",0
...,...,...,...
95,Wayne,"POLYGON ((2207573.855 561454.271, 2208904.812 ...",0
96,Wilkes,"POLYGON ((1249272.892 871985.581, 1255471.186 ...",0
97,Wilson,"POLYGON ((2239766.988 721791.129, 2240690.85 7...",0
98,Yadkin,"POLYGON ((1444174.565 852425.399, 1444356.629 ...",0


## Plotting turbines